In [1]:
import os
import json
import shutil
import subprocess
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from dotenv import load_dotenv
from pathlib import Path
from functools import wraps
from langchain.tools import tool
from langchain_openai import ChatOpenAI
load_dotenv()


# ===== LLM Client =====

class LLMClient:
    def __init__(self):
        self._client = ChatOpenAI(
            base_url=os.getenv("OPENAI_API_HOST"),
            api_key=os.getenv("OPENAI_API_KEY"),
            model="openai/gpt-oss-20b",
            timeout=None,
        )

    def generate(self, prompt: str) -> str:
        return self._client.invoke(prompt).content


# ===== Контекст (всё хранится в памяти, файлы только html/js/csv) =====

@dataclass
class AgentContext:
    requirements: str
    plan: list[dict] = field(default_factory=list)   # [{"action": ..., "is_done": bool, "executor": ...}]
    domain: str | None = None
    tests: str | None = None
    review: str | None = None
    # Только эти файлы пишутся на диск:
    files: dict[str, str] = field(default_factory=dict)  # {"index.html": "...", "app.js": "...", "data.csv": "..."}


# ===== Декоратор: логирование агента =====

def agent_step(name: str):
    def decorator(run_fn):
        @wraps(run_fn)
        def wrapper(self, ctx: AgentContext) -> AgentContext:
            print(f"\n▶ [{name}] started")
            ctx = run_fn(self, ctx)
            # Помечаем шаг выполненным в плане
            for step in ctx.plan:
                if step.get("executor") == name and not step["is_done"]:
                    step["is_done"] = True
                    break
            print(f"✅ [{name}] done")
            return ctx
        return wrapper
    return decorator


# ===== Декоратор: subprocess tool =====

def subprocess_tool(func):
    """Декоратор — запускает функцию через subprocess и возвращает stdout/stderr."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        command = func(*args, **kwargs)
        result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode != 0:
            print(f"⚠ subprocess error: {result.stderr[:300]}")
        return result.stdout or result.stderr
    return wrapper


# ===== Tools (langchain @tool) =====

@tool
def run_docker_compose(compose_path: str = "repo/docker-compose.yml") -> str:
    """Run docker compose up --build -d for given compose file path."""
    command = f"docker compose -f {compose_path} up --build -d".split()
    result = subprocess.run(command, capture_output=True, text=True)
    output = result.stdout + result.stderr
    if result.returncode == 0:
        return f"✅ Docker launched successfully.\n{output[:500]}"
    return f"❌ Docker failed (code {result.returncode}):\n{output[:500]}"


@tool
def check_docker_status() -> str:
    """Check running docker containers."""
    result = subprocess.run(["docker", "ps"], capture_output=True, text=True)
    return result.stdout


@tool
def write_output_files(files_json: str) -> str:
    """
    Write only .html, .js, .csv files to OUTPUT_DIR.
    files_json: JSON string like {"index.html": "<html>...", "app.js": "..."}
    """
    output_dir = Path(os.getenv("OUTPUT_DIR", "demo"))
    shutil.rmtree(output_dir, ignore_errors=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    files = json.loads(files_json)
    written = []
    for filename, content in files.items():
        ext = Path(filename).suffix.lower()
        if ext in (".html", ".js", ".csv"):
            (output_dir / filename).write_text(content, encoding="utf-8")
            written.append(filename)
        else:
            print(f"  ⏭ Skipped (not html/js/csv): {filename}")

    return f"Written to {output_dir}: {written}"


# ===== Базовый агент =====

class BaseAgent(ABC):
    def __init__(self, llm: LLMClient):
        self.llm = llm

    @abstractmethod
    def run(self, ctx: AgentContext) -> AgentContext:
        ...


# ===== 1. Менеджер — координирует, строит план =====

class ManagerAgent(BaseAgent):
    """Агент Менеджер: получает промт, составляет план, раздаёт задачи."""

    TOOLS = [run_docker_compose, check_docker_status, write_output_files]

    @agent_step("Manager")
    def run(self, ctx: AgentContext) -> AgentContext:
        prompt = f"""
Ты менеджер мультиагентной системы для создания веб-приложений.
Получи требования и составь план работы в виде JSON-массива.
Каждый шаг: {{"action": "описание", "is_done": false, "executor": "AgentName"}}

Допустимые executor: DDD, TDD, Developer, Reviewer, Executor

Требования: {ctx.requirements}

Верни ТОЛЬКО JSON-массив, без лишнего текста.
"""
        raw = self.llm.generate(prompt)
        # Чистим от markdown-обёртки если есть
        raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        try:
            ctx.plan = json.loads(raw)
        except json.JSONDecodeError:
            # Fallback план если LLM вернул некорректный JSON
            ctx.plan = [
                {"action": "Создай доменную модель", "is_done": False, "executor": "DDD"},
                {"action": "Создай тесты", "is_done": False, "executor": "TDD"},
                {"action": "Разработай веб-приложение (HTML/JS/CSS)", "is_done": False, "executor": "Developer"},
                {"action": "Проведи ревью кода", "is_done": False, "executor": "Reviewer"},
                {"action": "Запусти приложение", "is_done": False, "executor": "Executor"},
            ]
        print(f"  Plan: {json.dumps(ctx.plan, ensure_ascii=False, indent=2)}")
        return ctx


# ===== 2. DDD Агент =====

class DDDAgent(BaseAgent):
    """Анализирует доменную модель — результат хранится только в переменной."""

    @agent_step("DDD")
    def run(self, ctx: AgentContext) -> AgentContext:
        prompt = f"""
Опиши доменную модель (сущности, атрибуты, связи) для веб-приложения по требованиям:
{ctx.requirements}

Ответь структурированно, но кратко.
"""
        ctx.domain = self.llm.generate(prompt)
        print(f"  Domain model saved to ctx.domain (not written to disk)")
        return ctx


# ===== 3. TDD Агент =====

class TDDAgent(BaseAgent):
    """Генерирует тесты — хранятся только в переменной."""

    @agent_step("TDD")
    def run(self, ctx: AgentContext) -> AgentContext:
        prompt = f"""
Сгенерируй pytest-тесты для доменной модели:
{ctx.domain}

Требования: {ctx.requirements}

Верни только Python-код тестов.
"""
        ctx.tests = self.llm.generate(prompt)
        print(f"  Tests saved to ctx.tests (not written to disk)")
        return ctx


# ===== 4. Developer Агент =====

class DeveloperAgent(BaseAgent):
    """Генерирует HTML/JS/CSV файлы на основе промта. Только они пишутся на диск."""

    @agent_step("Developer")
    def run(self, ctx: AgentContext) -> AgentContext:

        html_prompt = f"""
Создай полноценное современное веб-приложение (одностраничное).
Требования: {ctx.requirements}
Доменная модель: {ctx.domain}

Верни ТОЛЬКО чистый HTML (включая inline CSS в <style> и подключение script.js).
Без markdown, без пояснений.
"""
        js_prompt = f"""
Создай JavaScript для веб-приложения:
Требования: {ctx.requirements}
Доменная модель: {ctx.domain}

Верни ТОЛЬКО чистый JS-код. Без markdown.
"""
        csv_prompt = f"""
Создай CSV с демонстрационными данными для веб-приложения:
Требования: {ctx.requirements}

Верни ТОЛЬКО CSV (с заголовками). Без markdown.
"""
        index_html = self.llm.generate(html_prompt)
        script_js = self.llm.generate(js_prompt)
        data_csv = self.llm.generate(csv_prompt)

        # Все файлы в памяти:
        ctx.files = {
            "index.html": index_html,
            "script.js": script_js,
            "data.csv": data_csv,
        }

        # Пишем на диск ТОЛЬКО html/js/csv через tool:
        files_json = json.dumps(ctx.files, ensure_ascii=False)
        result = write_output_files.invoke({"files_json": files_json})
        print(f"  {result}")
        return ctx


# ===== 5. Reviewer Агент =====

class ReviewerAgent(BaseAgent):
    """Code review — результат только в переменной."""

    @agent_step("Reviewer")
    def run(self, ctx: AgentContext) -> AgentContext:
        html_snippet = (ctx.files.get("index.html", "")[:800])
        js_snippet = (ctx.files.get("script.js", "")[:800])
        prompt = f"""
Проведи краткое code review для веб-приложения.
HTML (фрагмент): {html_snippet}
JS (фрагмент): {js_snippet}
Требования: {ctx.requirements}

Верни список конкретных замечаний и улучшений (до 10 пунктов).
"""
        ctx.review = self.llm.generate(prompt)
        print(f"  Review saved to ctx.review (not written to disk)")
        return ctx


# ===== 6. Executor Агент =====

class ExecutorAgent(BaseAgent):
    """Запускает docker compose через subprocess tool."""

    @agent_step("Executor")
    def run(self, ctx: AgentContext) -> AgentContext:
        compose_path = "repo/docker-compose.yml"
        if not Path(compose_path).exists():
            print(f"  ⚠ {compose_path} not found, skipping docker launch")
            return ctx
        result = run_docker_compose.invoke({"compose_path": compose_path})
        print(f"  {result[:300]}")
        return ctx


# ===== Pipeline =====

EXECUTOR_MAP = {
    "DDD":      DDDAgent,
    "TDD":      TDDAgent,
    "Developer": DeveloperAgent,
    "Reviewer": ReviewerAgent,
    "Executor": ExecutorAgent,
}

def run_pipeline(prompt: str, llm: LLMClient | None = None) -> AgentContext:
    if llm is None:
        llm = LLMClient()

    ctx = AgentContext(requirements=requirements)

    # Шаг 0: Менеджер строит план
    ctx = ManagerAgent(llm).run(ctx)

    # Шаги 1–N: каждый агент запускается только один раз
    seen = set()
    for step in ctx.plan:
        executor_name = step.get("executor")
        if executor_name in seen:
            print(f"  ⏭ {executor_name} уже выполнялся, пропускаем")
            continue
        seen.add(executor_name)
        AgentClass = EXECUTOR_MAP.get(executor_name)
        if AgentClass:
            ctx = AgentClass(llm).run(ctx)
        else:
            print(f"⚠ Unknown executor: {executor_name}, skipping")

    print("\n Pipeline complete!")
    print(f"   Files on disk : {list(ctx.files.keys())}")
    print(f"   Domain in mem : {bool(ctx.domain)}")
    print(f"   Tests in mem  : {bool(ctx.tests)}")
    print(f"   Review in mem : {bool(ctx.review)}")
    return ctx

# ===== Entry point =====


prompt = "Создай интерактивный дашборд с таблицей данных и графиком продаж."
run_pipeline(prompt)

/Applications/anaconda3/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [2]:
!env | grep OPEN

In [3]:
load_dotenv()

False

In [4]:
!pwd

/Users/annatezelnikova
